# Windows Event Log Monitor
**Author:** Mahitha Madhava Das  
**Date:** May 2026  
**Tools:** Python, PowerShell, subprocess  

## About This Project
This script automates the process of querying Windows Event Logs 
for critical system and application errors, simulating the production 
monitoring workflows used in enterprise application support environments.

This replicates what I did manually using Dynatrace and Splunk at 
Cognizant Technology Solutions when supporting Liberty Mutual Insurance.

## Skills Demonstrated
- Python scripting and automation
- Windows system administration
- Incident report generation
- Production monitoring concepts

In [3]:
# Import required libraries
import subprocess
import datetime
import os
import platform

# Check we are running on Windows
print(f"Operating System: {platform.system()}")
print(f"Report generated at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)
print("Log Monitor initialized successfully!")

Operating System: Windows
Report generated at: 2026-05-02 14:52:00
Log Monitor initialized successfully!


In [4]:
def get_windows_logs(log_name, entry_type, newest=10):
    """
    Query Windows Event Log for specific entry types.
    
    Args:
        log_name: 'System' or 'Application'
        entry_type: 'Error', 'Warning', or 'Information'
        newest: how many recent entries to retrieve
    
    Returns:
        String output of log entries
    """
    
    cmd = f"""
    Get-EventLog -LogName {log_name} -EntryType {entry_type} -Newest {newest} |
    Select-Object TimeGenerated, EntryType, Source, EventID, Message |
    Format-Table -AutoSize -Wrap
    """
    
    try:
        result = subprocess.run(
            ["powershell", "-Command", cmd],
            capture_output=True,
            text=True,
            timeout=30
        )
        
        if result.stdout.strip():
            return result.stdout
        else:
            return f"No {entry_type} entries found in {log_name} log.\n"
            
    except subprocess.TimeoutExpired:
        return f"Timeout while querying {log_name} log.\n"
    except Exception as e:
        return f"Error querying {log_name} log: {str(e)}\n"

print("Function defined successfully!")

Function defined successfully!


In [5]:
print("=" * 60)
print("       WINDOWS EVENT LOG MONITOR REPORT")
print("=" * 60)
print(f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

# Check System log errors
print("\n[ SYSTEM LOG — ERRORS (Last 10) ]")
print("-" * 60)
system_errors = get_windows_logs("System", "Error", 10)
print(system_errors)

# Check Application log errors
print("\n[ APPLICATION LOG — ERRORS (Last 10) ]")
print("-" * 60)
app_errors = get_windows_logs("Application", "Error", 10)
print(app_errors)

# Check System warnings
print("\n[ SYSTEM LOG — WARNINGS (Last 5) ]")
print("-" * 60)
system_warnings = get_windows_logs("System", "Warning", 5)
print(system_warnings)

print("=" * 60)
print("Log check complete.")

       WINDOWS EVENT LOG MONITOR REPORT
Generated: 2026-05-02 14:52:53

[ SYSTEM LOG — ERRORS (Last 10) ]
------------------------------------------------------------

TimeGenerated       EntryType Source                                EventID Message                                    
-------------       --------- ------                                ------- -------                                    
02-05-2026 14:26:58     Error Microsoft-Windows-TPM-WMI                1801 Secure Boot certificates have been updated 
                                                                            but are not yet applied to the device      
                                                                            firmware. Review the published guidance to 
                                                                            complete the update and ensure full        
                                                                            protection. This device signature   

In [13]:
def get_error_count(log_name, entry_type, newest=100):
    """Count entries for summary statistics."""
    cmd = f"""
    (Get-EventLog -LogName {log_name} -EntryType {entry_type} -Newest {newest}).Count
    """
    try:
        result = subprocess.run(
            ["powershell", "-Command", cmd],
            capture_output=True, text=True, timeout=30
        )
        count = result.stdout.strip()
        return int(count) if count.isdigit() else 0
    except:
        return 0

# Print summary
print("=" * 60)
print("           SUMMARY STATISTICS")
print("=" * 60)

sys_errors   = get_error_count("System", "Error")
app_errors   = get_error_count("Application", "Error")
sys_warnings = get_error_count("System", "Warning")

print(f"System Errors    (last 100 checked): {sys_errors}")
print(f"Application Errors (last 100 checked): {app_errors}")
print(f"System Warnings  (last 100 checked): {sys_warnings}")
print("-" * 60)

total = sys_errors + app_errors
if total == 0:
    print("STATUS: HEALTHY — No critical errors detected")
elif total <= 5:
    print("STATUS: LOW RISK — Minor errors present, monitor closely")
elif total <= 20:
    print("STATUS: MODERATE — Investigate error sources")
else:
    print("STATUS: HIGH ALERT — Significant errors detected, escalate")

print("=" * 60)

           SUMMARY STATISTICS
System Errors    (last 100 checked): 100
Application Errors (last 100 checked): 100
System Warnings  (last 100 checked): 100
------------------------------------------------------------
STATUS: HIGH ALERT — Significant errors detected, escalate


In [12]:
def get_error_count(log_name, entry_type, hours=24):
    """
    Count entries from the LAST 24 HOURS only — more meaningful than a raw count.
    """
    cmd = f"""
    $cutoff = (Get-Date).AddHours(-{hours})
    $entries = Get-EventLog -LogName {log_name} -EntryType {entry_type} -After $cutoff -ErrorAction SilentlyContinue
    if ($entries) {{ $entries.Count }} else {{ 0 }}
    """
    try:
        result = subprocess.run(
            ["powershell", "-Command", cmd],
            capture_output=True, text=True, timeout=30
        )
        count = result.stdout.strip()
        return int(count) if count.isdigit() else 0
    except:
        return 0

# Print summary
print("=" * 60)
print("        SUMMARY — LAST 24 HOURS")
print("=" * 60)

sys_errors   = get_error_count("System", "Error")
app_errors   = get_error_count("Application", "Error")
sys_warnings = get_error_count("System", "Warning")

print(f"System Errors      (last 24 hrs): {sys_errors}")
print(f"Application Errors (last 24 hrs): {app_errors}")
print(f"System Warnings    (last 24 hrs): {sys_warnings}")
print("-" * 60)

total = sys_errors + app_errors
if total == 0:
    print("STATUS: HEALTHY -- No critical errors in last 24 hours")
elif total <= 5:
    print("STATUS: LOW RISK -- Minor errors present, monitor closely")
elif total <= 20:
    print("STATUS: MODERATE -- Investigate error sources")
else:
    print("STATUS: HIGH ALERT -- Significant errors, investigate")

print("=" * 60)
print("\nNOTE: Some errors are normal on any Windows machine.")
print("Key is to identify RECURRING sources and investigate those.")

        SUMMARY — LAST 24 HOURS
System Errors      (last 24 hrs): 11
Application Errors (last 24 hrs): 1
System Warnings    (last 24 hrs): 78
------------------------------------------------------------
STATUS: MODERATE -- Investigate error sources

NOTE: Some errors are normal on any Windows machine.
Key is to identify RECURRING sources and investigate those.


In [7]:
def generate_report():
    """
    Generate a full incident report and save it to a text file.
    """
    
    timestamp = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    report_filename = f"log_report_{timestamp}.txt"
    
    # Build report content
    report_lines = []
    report_lines.append("=" * 60)
    report_lines.append("     WINDOWS EVENT LOG — INCIDENT REPORT")
    report_lines.append("=" * 60)
    report_lines.append(f"Report generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report_lines.append(f"Hostname: {os.environ.get('COMPUTERNAME', 'Unknown')}")
    report_lines.append("=" * 60)
    
    # System errors
    report_lines.append("\n[ SYSTEM ERRORS — Last 10 ]")
    report_lines.append("-" * 60)
    report_lines.append(get_windows_logs("System", "Error", 10))
    
    # Application errors
    report_lines.append("\n[ APPLICATION ERRORS — Last 10 ]")
    report_lines.append("-" * 60)
    report_lines.append(get_windows_logs("Application", "Error", 10))
    
    # System warnings
    report_lines.append("\n[ SYSTEM WARNINGS — Last 5 ]")
    report_lines.append("-" * 60)
    report_lines.append(get_windows_logs("System", "Warning", 5))
    
    report_lines.append("\n" + "=" * 60)
    report_lines.append("END OF REPORT")
    report_lines.append("=" * 60)
    
    # Save to file
    full_report = "\n".join(report_lines)
    
    with open(report_filename, "w", encoding="utf-8") as f:
        f.write(full_report)
    
    print(f"Report saved successfully: {report_filename}")
    print(f"File location: {os.path.abspath(report_filename)}")
    return report_filename

# Run the report generator
saved_file = generate_report()

Report saved successfully: log_report_2026-05-02_14-54-02.txt
File location: C:\Users\dasma\Documents\log_report_2026-05-02_14-54-02.txt


In [11]:
def get_top_error_sources(log_name, entry_type, hours=24, top=5):
    """
    Show WHICH sources are generating the most errors — 
    this is what a real L2 analyst looks at first.
    """
    cmd = f"""
    $cutoff = (Get-Date).AddHours(-{hours})
    Get-EventLog -LogName {log_name} -EntryType {entry_type} -After $cutoff -ErrorAction SilentlyContinue |
    Group-Object Source |
    Sort-Object Count -Descending |
    Select-Object -First {top} |
    Format-Table Count, Name -AutoSize
    """
    try:
        result = subprocess.run(
            ["powershell", "-Command", cmd],
            capture_output=True, text=True, timeout=30
        )
        return result.stdout if result.stdout.strip() else "No entries found.\n"
    except Exception as e:
        return f"Error: {str(e)}\n"

print("TOP ERROR SOURCES — SYSTEM LOG (last 24 hrs)")
print("-" * 60)
print(get_top_error_sources("System", "Error"))

print("TOP ERROR SOURCES — APPLICATION LOG (last 24 hrs)")
print("-" * 60)
print(get_top_error_sources("Application", "Error"))

TOP ERROR SOURCES — SYSTEM LOG (last 24 hrs)
------------------------------------------------------------

Count Name                                 
----- ----                                 
    5 Microsoft-Windows-WindowsUpdateClient
    3 Service Control Manager              
    1 Microsoft-Windows-NDIS               
    1 EventLog                             
    1 Microsoft-Windows-TPM-WMI            



TOP ERROR SOURCES — APPLICATION LOG (last 24 hrs)
------------------------------------------------------------

Count Name            
----- ----            
    1 Application Hang





## Results & Observations

### Run environment
- **Date:** May 2026
- **Machine:** Personal Windows laptop (test environment)
- **Script runtime:** ~15 seconds

### Summary findings

| Log | Type | Count (last 24 hrs) | Assessment |
|---|---|---|---|
| System | Errors | 11 | Normal range for Windows |
| Application | Errors | 1 | Stable — low concern |
| System | Warnings | 78 | Elevated — warrants investigation |

**Overall status: MODERATE**

### Analysis

The 78 system warnings is the most significant finding.
In a production environment, a count this high would
immediately trigger source analysis to determine whether
a single component is generating repeated warnings
(a known pattern called a "warning storm") or whether
warnings are spread across multiple sources.

The 11 system errors fall within the normal range
for a personal Windows machine — background services,
driver events, and scheduled task failures commonly
generate this level of noise.

The single application error indicates applications
are running stably, which in a production context
would be a positive signal for the application layer.

### What a real L2 analyst does next

In an enterprise environment (as I did at Cognizant
supporting Liberty Mutual Insurance), the next step
after this summary would be to:

1. Run source grouping to identify which component
   is generating the most warnings
2. Cross-reference EventIDs against known issue
   runbooks
3. If a new source pattern is found, open a Problem
   ticket in ServiceNow for root cause analysis
4. Document findings and update the knowledge base

This script automates Step 1 of that workflow.

### Production equivalent

This script replicates the manual log-checking process
I performed using Dynatrace and Splunk at Cognizant.
In production, this would be:
- Scheduled via Windows Task Scheduler (every hour)
- Output piped to a monitoring dashboard
- Critical thresholds set to auto-create ServiceNow
  incidents
- Results stored in a database for 30-day trend analysis